<a href="https://colab.research.google.com/github/Teja3993/Deep-Learning-Lab-exercises/blob/main/Experiment_6_Predict_words_using_Autoencoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Create a Toy Dataset
# We want the model to predict the missing word (represented by "MASK")
inputs = [
    "the MASK is blue",
    "the apple is MASK",
    "MASK sky is blue"
]
targets = [
    "the sky is blue",
    "the apple is red",
    "the sky is blue"
]

# 2. Tokenize the Words (Convert text to numbers)
tokenizer = Tokenizer(filters='') # Keep all characters so "MASK" stays intact
tokenizer.fit_on_texts(inputs + targets)
vocab_size = len(tokenizer.word_index) + 1 # +1 for padding (0)

# Convert sentences to integer sequences
X_seq = tokenizer.texts_to_sequences(inputs)
y_seq = tokenizer.texts_to_sequences(targets)

# Pad sequences so they are all the same length (4 words here)
max_length = 4
X_padded = pad_sequences(X_seq, maxlen=max_length, padding='post')
y_padded = pad_sequences(y_seq, maxlen=max_length, padding='post')

# The output layer needs target data in 3D shape: (samples, time_steps, features)
# We one-hot encode the target words for the categorical crossentropy loss
y_one_hot = tf.keras.utils.to_categorical(y_padded, num_classes=vocab_size)

# 3. Build the Encoder-Decoder Autoencoder Model
model = Sequential()

# --- ENCODER ---
# Embedding layer learns word representations (turns integers into dense vectors)
model.add(Embedding(input_dim=vocab_size, output_dim=16, input_length=max_length))

# LSTM compresses the input sequence into a single "thought vector" (size 32)
model.add(LSTM(32, activation='relu'))

# --- BRIDGE ---
# RepeatVector takes that single thought vector and copies it for every word we need to generate
model.add(RepeatVector(max_length))

# --- DECODER ---
# LSTM processes the repeated vectors to reconstruct the sequence
model.add(LSTM(32, activation='relu', return_sequences=True))

# TimeDistributed applies the final Dense layer to every time step independently
# Softmax gives us the probability distribution over our vocabulary for each word position
model.add(TimeDistributed(Dense(vocab_size, activation='softmax')))

# 4. Compile the Model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 5. Train the Model
print("Training the Autoencoder...")
model.fit(X_padded, y_one_hot, epochs=200, verbose=0) # verbose=0 to hide epoch spam
print("Training complete!\n")

# 6. Test the Model (Predicting the MASKed words)
print("--- Predictions ---")
predictions = model.predict(X_padded, verbose=0)

# Helper function to convert numeric predictions back to words
def decode_sequence(pred_seq):
    predicted_word_indices = np.argmax(pred_seq, axis=-1)
    words = []
    for index in predicted_word_indices:
        if index != 0: # Ignore padding
            # Find the word corresponding to the index
            word = list(tokenizer.word_index.keys())[list(tokenizer.word_index.values()).index(index)]
            words.append(word)
    return " ".join(words)

# Display results
for i in range(len(inputs)):
    pred_text = decode_sequence(predictions[i])
    print(f"Input   : {inputs[i]}")
    print(f"Predicted: {pred_text}")
    print(f"Target  : {targets[i]}\n")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Training the Autoencoder...
Training complete!

--- Predictions ---
Input   : the MASK is blue
Predicted: the sky is blue
Target  : the sky is blue

Input   : the apple is MASK
Predicted: the apple is red
Target  : the apple is red

Input   : MASK sky is blue
Predicted: the sky is blue
Target  : the sky is blue

